# 03 - GPT Text Formatting

## Purpose

This notebook improves the quality of the extracted PDF text before embedding.

PDF extraction can produce text with broken lines, inconsistent spacing, missing punctuation, or awkward formatting. This notebook uses a GPT model to clean and format the extracted text while preserving the original meaning.

The cleaned documents will be used in the next step for token-based splitting and vector indexing.

## Main Changes in Version 2

This version adds:

- GPT-based text formatting
- Cleaner input before embedding
- Metadata preservation
- Better preparation for semantic search

## Input

This notebook reloads the uploaded PDF and recreates the structured document list:

- `docs_list_md_split`

## Main Outputs

- `formatted_docs_list`
- `string_list_formatted`

## Notebook Flow

```text
PDF text
        ↓
Header-based document structure
        ↓
GPT formatting prompt
        ↓
Cleaned document text
        ↓
Formatted LangChain documents
        ↓
Ready for token splitting

In [0]:
%pip install langchain langchain-community langchain-openai pypdf tiktoken chromadb

In [0]:
dbutils.library.restartPython()

In [0]:
import os

os.environ["OPENAI_API_KEY"] = "API_KEY_Here"

In [0]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter

pdf_path = "/Volumes/workspace/365pdf/365pdfupload/Introduction_to_Tableau.pdf"

loader_pdf = PyPDFLoader(pdf_path)
docs_list = loader_pdf.load()

markdown_transcript = ""

for doc in docs_list:
    page_number = doc.metadata.get("page", "unknown")
    page_text = doc.page_content

    markdown_transcript += f"\n\n# Introduction to Tableau\n"
    markdown_transcript += f"## Page {page_number}\n\n"
    markdown_transcript += page_text

headers_to_split_on = [
    ("#", "Section Title"),
    ("##", "Page Title")
]

md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)

docs_list_md_split = md_splitter.split_text(markdown_transcript)

print("Header-split documents recreated:", len(docs_list_md_split))

for i, doc in enumerate(docs_list_md_split[:2]):
    print(f"\nDocument {i}")
    print("Metadata:", doc.metadata)
    print(doc.page_content[:500])
    print("-" * 80)

In [0]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

chat = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

formatting_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a document-cleaning assistant.

Your task is to clean text extracted from a PDF.

Rules:
- Fix broken line breaks and awkward spacing.
- Improve punctuation and capitalization where needed.
- Keep the original meaning unchanged.
- Do not add new information.
- Do not summarize.
- Do not remove important details.
- Return only the cleaned text.
"""
    ),
    (
        "human",
        """
Clean and format the following extracted PDF text:

{text}
"""
    )
])

formatting_chain = formatting_prompt | chat | StrOutputParser()

print("GPT formatting chain created")

In [0]:
test_text = docs_list_md_split[0].page_content[:2000]

formatted_test = formatting_chain.invoke({
    "text": test_text
})

print("Original preview:")
print(test_text[:1000])

print("\n" + "=" * 80)
print("Formatted preview:")
print(formatted_test[:1000])

In [0]:
from langchain_core.documents import Document
import time

formatted_docs_list = []

for i, doc in enumerate(docs_list_md_split):
    print(f"Formatting document {i + 1} of {len(docs_list_md_split)}")

    text_to_format = doc.page_content[:3000]

    formatted_text = formatting_chain.invoke({
        "text": text_to_format
    })

    formatted_doc = Document(
        page_content=formatted_text,
        metadata=doc.metadata
    )

    formatted_docs_list.append(formatted_doc)

    time.sleep(0.5)

print("Formatting complete")
print("Number of formatted documents:", len(formatted_docs_list))

In [0]:
for i, doc in enumerate(formatted_docs_list[:3]):
    print(f"\nFormatted Document {i}")
    print("Metadata:", doc.metadata)
    print("Content preview:")
    print(doc.page_content[:800])
    print("-" * 80)

In [0]:
string_list_formatted = "\n\n".join(
    [doc.page_content for doc in formatted_docs_list]
)

print("Combined formatted text created")
print("Total characters:", len(string_list_formatted))
print(string_list_formatted[:1000])